# Recreating Habit Tracking Processing from R to Python  
Katy Yut  
May 31, 2025

In [ ]:
# Load packages
from googleapiclient.discovery import build
from google.oauth2 import service_account
import io
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from datetime import datetime, timedelta
import seaborn as sns
import os
# from wordcloud import WordCloud
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
# plt.style.use('seaborn')
sns.set_palette("deep")



from habit_tracking import config

In [ ]:
class HabitTracker:
    def __init__(self):
        self.df = None
        self.sleep_data = None
        self.weight_data = None

    def load_google_sheets_data(self, service_account_file, spreadsheet_id):
        """Load data from Google Sheets"""
        credentials = service_account.Credentials.from_service_account_file(
            service_account_file,
            scopes=['https://www.googleapis.com/auth/drive.readonly']
        )
        
        service = build('drive', 'v3', credentials=credentials)
        request = service.files().export_media(
            fileId=spreadsheet_id,
            mimeType='text/csv'
        )
        
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            
        fh.seek(0)
        return pd.read_csv(fh)

    def apply_cleaning(self, service_account_file, spreadsheet_id, sleep_file=None, weight_file=None):
        """Load and clean all data sources"""
        # Load main tracking data
        df_raw = self.load_google_sheets_data(service_account_file, spreadsheet_id)
        
        # Clean column names
        self.df = df_raw.rename(columns=config.CLEAN_COLUMN_NAMES)
        
        # Cleaning and processing functions
        self.clean_dates()
        self.process_boolean_variables()
        self.process_categorical_variables()
        self.calculate_tracked_habits()
        # period_dates = self.process_periods() # DO SOMETHING WITH THIS--RETURN OR ADD TO SELF __INIT__
        
        # Load additional data if provided
        if sleep_file:
            self.sleep_data = pd.read_csv(sleep_file)
        if weight_file:
            self.weight_data = pd.read_csv(weight_file)
            self.clean_weight_data()
            
    def clean_dates(self):
        """Clean and process dates in the dataframe"""
        
        # Fill missing dates with Submission_DateTime
        self.df['Date'] = self.df.apply(
            lambda row: pd.to_datetime(row['Submission_DateTime'])
            if pd.isna(row['Date']) else pd.to_datetime(row['Date']),
            axis=1
        )

        # Convert from datetime to date
        self.df['Date'] = self.df['Date'].dt.date

        # Create complete date range
        date_range = pd.date_range(
            start=config.EXERCISE_START_DATE,  # Original start date from R code
            end=pd.Timestamp.today().date(),
            freq='D'
        )

        # Create a dataframe with all dates
        all_dates = pd.DataFrame({'Date': date_range})

        # Convert from datetime to date
        all_dates['Date'] = all_dates['Date'].dt.date

        # Merge with existing data
        self.df = pd.merge(all_dates, self.df, on='Date', how='left')
        
    def process_boolean_variables(self):
        """Convert Yes/No to boolean and handle missing values"""
        for col in config.BOOLEAN_VARIABLES:
            self.df[col] = self.df[col].map({'Yes': True, 'No': False})
            self.df[col] = self.df[col].astype('boolean')
            
    def process_categorical_variables(self):
        """Convert categorical variables to proper types"""
        
        # Libido as ordered category
        self.df['Libido'] = pd.Categorical(
            self.df['Libido'].map({'N/A': 'None', 'Moderate': 'Medium', 'High': 'High'}),
            categories=['None', 'Medium', 'High'],
            ordered=True
        )
        
        # Skin as ordered category
        self.df['Skin'] = pd.Categorical(
            self.df['Skin'],
            categories=['Major breakouts', 'Minor breakouts', 'Clear'],
            ordered=True
        )
        
    def calculate_tracked_habits(self):
        """Calculate whether habits were tracked each day"""
        self.df['Tracked_Habits'] = (
            pd.to_datetime(self.df['Submission_DateTime']).dt.date == 
            pd.to_datetime(self.df['Date']).dt.date
        )
        
        self.df['Collected_Data'] = ~self.df['Submission_DateTime'].isna()
        
    def process_sleep_data(self):
        """Process sleep data if available"""
        if self.sleep_data is None:
            return
            
        # Clean sleep data columns
        sleep_clean = self.sleep_data.rename(columns=config.CLEAN_SLEEP_COLUMNS)
        
        # Convert time columns
        sleep_clean['Time_in_Bed_hr'] = sleep_clean['Time_in_Bed_sec'] / 3600
        sleep_clean['Time_Asleep_hr'] = sleep_clean['Time_Asleep_sec'] / 3600
        sleep_clean['Time_Before_Sleep_hr'] = sleep_clean['Time_Before_Sleep_sec'] / 3600
        
        self.sleep_data = sleep_clean
        
    def clean_weight_data(self):
        """Clean and process the weight tracking data"""
        if self.weight_data is None:
            return False
            
        # Clean headers
        self.weight_data = self.weight_data.rename(columns=config.CLEAN_WEIGHT_COLUMNS)
        
        # Convert datetime
        self.weight_data['Submission_DateTime'] = pd.to_datetime(self.weight_data['Submission_DateTime'])
        self.weight_data['Date'] = self.weight_data['Submission_DateTime'].dt.date
        
        return True
    
    def process_periods(self):
        """Process period data to get start and end dates"""
        if self.df is None:
            return None
            
        # Create a copy of period data
        periods = self.df[['Date', 'Period']].copy()
        
        # Calculate changes in period status
        periods['Binary_Start_End'] = periods['Period'].fillna(0).diff()
        
        # Get start dates (where Binary_Start_End == 1)
        starts = periods[periods['Binary_Start_End'] == 1]['Date']
        
        # Get end dates (day before Binary_Start_End == -1)
        ends = periods[periods['Binary_Start_End'] == -1]['Date'].apply(
            lambda x: x - timedelta(days=1)
        )
        
        # Combine into DataFrame
        period_dates = pd.DataFrame({
            'start': starts.values,
            'end': ends.values
        })
        
        return period_dates
    
    def calculate_monthly_stats(self):
        """Calculate monthly statistics"""
        # Extract month and year
        self.df['Month'] = pd.to_datetime(self.df['Date']).dt.strftime('%m')
        self.df['Year'] = pd.to_datetime(self.df['Date']).dt.strftime('%Y')
        
        # Calculate raw counts
        monthly_stats_raw = self.df.groupby(['Year', 'Month'])[config.BOOLEAN_VARIABLES].sum()
        
        # Calculate percentages
        monthly_stats_percent = (
            self.df.groupby(['Year', 'Month'])[config.BOOLEAN_VARIABLES].sum() / 
            self.df.groupby(['Year', 'Month'])[config.BOOLEAN_VARIABLES].count() * 100
        )
        
        # Convert 'Mental_Health' to numeric (if not already)
        self.df['Mental_Health'] = pd.to_numeric(self.df['Mental_Health'], errors='coerce')
        
        # Add mental health average
        mental_health_avg = self.df.groupby(['Year', 'Month'])['Mental_Health'].mean()
        monthly_stats_raw['Mental_Health'] = mental_health_avg
        monthly_stats_percent['Mental_Health'] = mental_health_avg * 10
        
        return monthly_stats_raw, monthly_stats_percent
        
    def plot_cumulative_habits(self):
        """Plot cumulative habits over time"""
        plt.figure(figsize=(15, 8))
        
        # Plot each variable
        for var in config.BOOLEAN_VARIABLES:
            cumsum = self.df[var].fillna(False).cumsum()
            plt.plot(self.df['Date'], cumsum, 
                    label=var, color=config.VAR_COLORS.get(var, 'gray'))
            
        plt.title('Cumulative Habits Over Time')
        plt.xlabel('Date')
        plt.ylabel('Number of Days')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        
    def plot_mental_health_trend(self):
        """Plot mental health trend over time"""
        plt.figure(figsize=(15, 6))
        
        # Convert Mental_Health to numeric
        mental_health_numeric = pd.to_numeric(self.df['Mental_Health'], errors='coerce')
        
        # Plot the trend
        plt.plot(self.df['Date'], mental_health_numeric, alpha=0.5)
        
        # Add smoothed trend line
        window_size = 30  # Adjust as needed
        rolling_mean = mental_health_numeric.rolling(window=window_size, center=True).mean()
        plt.plot(self.df['Date'], rolling_mean, color='red', linewidth=2)
        
        plt.title('Mental Health Over Time')
        plt.xlabel('Date')
        plt.ylabel('Mental Health Score (1-10)')
        plt.grid(True, alpha=0.3)
        
    # OTHER VERSION --------------------------------------------------------------
    def plot_monthly_heatmap(self, start_date=None):
        """Plot a heatmap of monthly statistics"""
        monthly_stats, monthly_stats_percent = self.calculate_monthly_stats()

        monthly_stats_percent.reset_index(inplace=True)

        # Create year-month column
        monthly_stats_percent['Year_Month'] = monthly_stats_percent.apply(
            lambda x: f"{int(x['Year'])}-{int(x['Month']):02d}", axis=1
        )

        # Filter by start date if provided
        if start_date:
            monthly_stats_percent = monthly_stats_percent[
                monthly_stats_percent['Year_Month'] >= start_date
            ]

        # Prepare data for heatmap
        heatmap_data = monthly_stats_percent.melt(
            id_vars=['Year_Month'],
            value_vars=config.BOOLEAN_VARIABLES + ['Mental_Health'],
            var_name='Variable',
            value_name='Percentage'
        )

        # Create heatmap
        plt.figure(figsize=(15, 10))
        pivot_table = heatmap_data.pivot(
            index='Year_Month',
            columns='Variable',
            values='Percentage'
        )

        # Replace NaN with 0
        # pivot_table.replace(np.nan, 0, inplace=True)
        pivot_table = pivot_table.astype(float).fillna(0)

        sns.heatmap(
            pivot_table,
            cmap='magma_r',
            cbar_kws={'label': 'Percentage'},
            xticklabels=True,
            yticklabels=True
        )

        plt.title('Monthly Percentages Heatmap')
        plt.xticks(rotation=90)
        plt.tight_layout()
        return plt.gcf()
    
    def plot_sleep_pattern(self, sleep_time=None):
        """Plot sleep patterns over time"""
        if sleep_time is None:
            sleep_time = self.process_sleep_data()
            
        if sleep_time is None:
            return None
            
        plt.figure(figsize=(15, 10))
        
        # Plot sleep segments
        plt.hlines(
            y=sleep_time['Date'],
            xmin=sleep_time['Sleep_Start_Time'],
            xmax=sleep_time['Sleep_End_Time'],
            color='navy',
            alpha=0.3,
            linewidth=2
        )
        
        plt.title('Sleep Patterns Over Time')
        plt.xlabel('Time of Day (Hours)')
        plt.ylabel('Date')
        
        # Set x-axis limits to show 24-hour period
        plt.xlim(20, 32)  # Show 8pm to 8am next day
        
        # Custom x-axis labels
        xticks = range(20, 33, 2)
        xtick_labels = [f"{h%24:02d}:00" for h in xticks]
        plt.xticks(xticks, xtick_labels)
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        return plt.gcf()
        
    def plot_sleep_quality(self):
        """Plot average monthly sleep quality over time"""
        if self.sleep_data is None:
            return None
            
        # Convert sleep quality to numeric (remove % sign)
        self.sleep_data['Sleep_Quality_Num'] = self.sleep_data['Sleep_Quality'].str.rstrip('%').astype(float)
        
        # Create month column
        self.sleep_data['Year_Month'] = pd.to_datetime(self.sleep_data['Sleep_End']).dt.strftime('%Y-%m')
        
        # Calculate monthly averages
        monthly_quality = self.sleep_data.groupby('Year_Month')['Sleep_Quality_Num'].mean().reset_index()
        monthly_quality['Date'] = pd.to_datetime(monthly_quality['Year_Month'] + '-01')
        
        plt.figure(figsize=(12, 6))
        plt.plot(monthly_quality['Date'], monthly_quality['Sleep_Quality_Num'], 
                 marker='o', linestyle='-', color='navy')
        
        plt.title('Average Monthly Sleep Quality')
        plt.xlabel('Month')
        plt.ylabel('Sleep Quality (%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        return plt.gcf()
        
    def plot_weight_trends(self):
        """Plot weight trends over time with a smoothed line"""
        if self.weight_data is None:
            return None
            
        plt.figure(figsize=(12, 6))
        
        # Plot actual measurements
        plt.scatter(
            self.weight_data['Date'],
            self.weight_data['Weight_lb'],
            color='grey',
            alpha=0.5,
            s=20,
            label='Measurements'
        )
        
        # Calculate and plot trend line
        x = np.array([(d - self.weight_data['Date'].min()).days 
                      for d in self.weight_data['Date']])
        y = self.weight_data['Weight_lb'].values
        
        # Fit a polynomial
        z = np.polyfit(x, y, 5)
        p = np.poly1d(z)
        
        # Generate points for smooth curve
        x_smooth = np.linspace(x.min(), x.max(), 300)
        y_smooth = p(x_smooth)
        
        # Convert x_smooth back to dates
        dates_smooth = [self.weight_data['Date'].min() + timedelta(days=int(d)) 
                       for d in x_smooth]
        
        plt.plot(dates_smooth, y_smooth, 'navy', label='Trend')
        
        plt.title('Weight Trends Over Time')
        plt.xlabel('Date')
        plt.ylabel('Weight (lb)')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        
        return plt.gcf()
        
    def plot_body_composition(self):
        """Plot body composition trends over time"""
        if self.weight_data is None:
            return None
            
        # Select body composition columns
        composition_cols = [
            'Body_Fat', 'Body_Water', 'Skeletal_Muscle',
            'Subcutaneous_Fat', 'Visceral_Fat'
        ]
        
        # Create long format data
        comp_data = self.weight_data.melt(
            id_vars=['Date'],
            value_vars=composition_cols,
            var_name='Metric',
            value_name='Percentage'
        )
        
        plt.figure(figsize=(12, 6))
        
        # Plot each metric
        for metric in composition_cols:
            metric_data = comp_data[comp_data['Metric'] == metric]
            plt.plot(
                metric_data['Date'],
                metric_data['Percentage'],
                label=metric.replace('_', ' '),
                alpha=0.7,
                marker='o',
                markersize=3
            )
        
        plt.title('Body Composition Trends')
        plt.xlabel('Date')
        plt.ylabel('Percentage')
        plt.grid(True, alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        
        return plt.gcf()
        
    def plot_goal_heatmap(self, start_date=None):
        """Plot a heatmap showing goal achievement"""
        monthly_stats = self.calculate_monthly_stats()
        
        # Create year-month column
        monthly_stats['Year_Month'] = monthly_stats.apply(
            lambda x: f"{int(x['Year'])}-{int(x['Month']):02d}", axis=1
        )
        
        # Filter by start date if provided
        if start_date:
            monthly_stats = monthly_stats[
                monthly_stats['Year_Month'] >= start_date
            ]
        
        # Process positive goals
        pos_goals_data = []
        for var, goal in self.positive_goals.items():
            if var in monthly_stats.columns:
                monthly_stats[f'{var}_Achieved'] = monthly_stats[var] > goal
                pos_goals_data.append({
                    'Variable': var,
                    'Goal': goal,
                    'Type': 'positive'
                })
                
        # Process negative goals
        neg_goals_data = []
        for var, goal in self.negative_goals.items():
            if var in monthly_stats.columns:
                monthly_stats[f'{var}_Achieved'] = monthly_stats[var] < goal
                neg_goals_data.append({
                    'Variable': var,
                    'Goal': goal,
                    'Type': 'negative'
                })
                
        # Combine goal data
        goals_data = pd.DataFrame(pos_goals_data + neg_goals_data)
        
        # Prepare data for heatmap
        achieved_cols = [col for col in monthly_stats.columns if col.endswith('_Achieved')]
        heatmap_data = monthly_stats.melt(
            id_vars=['Year_Month'],
            value_vars=achieved_cols,
            var_name='Variable',
            value_name='Achieved'
        )
        
        # Clean variable names
        heatmap_data['Variable'] = heatmap_data['Variable'].str.replace('_Achieved', '')
        
        # Create heatmap
        plt.figure(figsize=(15, 10))
        pivot_table = heatmap_data.pivot(
            index='Year_Month',
            columns='Variable',
            values='Achieved'
        )
        
        sns.heatmap(
            pivot_table,
            cmap=['white', 'green'],
            cbar=False,
            xticklabels=True,
            yticklabels=True
        )
        
        plt.title('Monthly Goal Achievement')
        plt.xticks(rotation=90)
        plt.tight_layout()
        return plt.gcf()
        
    def plot_habit_timeline(self, start_date=None):
        """Plot a GitHub-style daily tracker for habits"""
        if self.df is None:
            return None
            
        # Convert boolean columns to numeric
        habit_data = self.df.copy()
        for col in config.BOOLEAN_VARIABLES:
            if col in habit_data.columns:
                habit_data[col] = habit_data[col].fillna(False) # replace NA with 0
                habit_data[col] = habit_data[col].astype(int) # convert from boolean to int

        # Add period data
        period_data = habit_data[['Date', 'Period']].copy()
        period_data['Variable'] = 'Period'
        period_data = period_data.rename(columns={'Period': 'Value'})

        # Prepare habit data
        habit_data = habit_data.melt(
            id_vars=['Date'],
            value_vars=config.BOOLEAN_VARIABLES,
            var_name='Variable',
            value_name='Value'
        )

        # Combine data
        all_data = pd.concat([habit_data, period_data])

        # Apply color coding
        def get_value_category(row):
            if row['Variable'] in ['Math', 'Sex', 'O', 'Period']:
                return -2 if row['Value'] == 1 else 0
            elif row['Variable'] in ['Caffeine', 'Alcohol', 'Weed', 'Delta8']:
                return -1 if row['Value'] == 1 else 0
            else:
                return 1 if row['Value'] == 1 else 0
                
        all_data['Category'] = all_data.apply(get_value_category, axis=1)

        # Filter by start date
        if start_date:
            all_data[all_data['Date'] >= pd.to_datetime(start_date).date()]

        # Create heatmap
        plt.figure(figsize=(15, 10))

        pivot_table = all_data.pivot_table(
            index='Date',
            columns='Variable',
            values='Category',
            aggfunc='sum'
        )

        # Custom colormap
        colors = ['pink', 'red', 'white', 'green']
        cmap = LinearSegmentedColormap.from_list('custom', colors)

        sns.heatmap(
            pivot_table,
            cmap=cmap,
            cbar=True,
            xticklabels=True,
            yticklabels=True,
            center=0
        )

        plt.title('Daily Habit Tracker')
        plt.xticks(rotation=90)
        plt.tight_layout()
        # return plt.gcf()


# main.py

In [ ]:
tracker = HabitTracker()

local_filepath = "/Users/katyyut/Desktop/01_Code/GitHub/habit-tracking/"
    
# Load data
SERVICE_ACCOUNT_FILE = f'{local_filepath}habit-tracking-python-379e261c6b6d.json'
SPREADSHEET_ID = '1LyIl8YRyw8nwP2TWBFAw-oRvDU9xnqj1BRVglntLNrU'

df_raw = tracker.load_google_sheets_data(SERVICE_ACCOUNT_FILE, SPREADSHEET_ID)

tracker.apply_cleaning(
    service_account_file=SERVICE_ACCOUNT_FILE,
    spreadsheet_id=SPREADSHEET_ID,
    sleep_file=f'{local_filepath}data/raw/20240716_Sleep_Data.csv',
    weight_file=f'{local_filepath}data/raw/20230911_Weight_Data.csv'
)

In [ ]:
tracker.df

In [ ]:
# MOVED THIS INSIDE load_data()
# Process data
# tracker.clean_dates()
# tracker.process_boolean_variables()
# tracker.process_categorical_variables()
# tracker.calculate_tracked_habits()

# Calculate statistics
monthly_raw, monthly_percent = tracker.calculate_monthly_stats()

In [ ]:
# Generate visualizations
tracker.plot_cumulative_habits()
tracker.plot_mental_health_trend()

plt.show() 

In [ ]:
# Process and plot weight data
tracker.clean_weight_data()

# Create all plots
plt.figure(1)
tracker.plot_sleep_pattern()

plt.figure(2)
tracker.plot_sleep_quality()

plt.figure(3)
tracker.plot_weight_trends()

plt.figure(4)
tracker.plot_body_composition()

plt.figure(5)
tracker.plot_monthly_heatmap("2022-06")

# plt.figure(6)
# tracker.plot_goal_heatmap("2022-06")

plt.figure(7)
tracker.plot_habit_timeline(config.HT_START_DATE)

plt.show()

# Scratch

In [ ]:
# Constants
SERVICE_ACCOUNT_FILE = '/Users/katyyut/Desktop/01_Code/GitHub/habit_tracking/02_Python_Code/habit-tracking-python-187fa005703f.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
FILE_ID = '1LyIl8YRyw8nwP2TWBFAw-oRvDU9xnqj1BRVglntLNrU'  # The Google Sheet's file ID

# Authenticate
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('drive', 'v3', credentials=creds)

# Request export as CSV
request = service.files().export_media(fileId=FILE_ID,
                                       mimeType='text/csv')

# Download the CSV content
fh = io.BytesIO()
downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    status, done = downloader.next_chunk()

# Load into pandas
fh.seek(0)
df_raw = pd.read_csv(fh)

# print(df_raw)

In [ ]:
df_raw